# Семинар 8. Итоги пилота: эксперимент и квазиэксперимент

Легенда:
пилот «Кэшбэк 5 % на доставку» закончился 27 сентября. Дизайн мы зафиксировали на семинаре 7:
три группы — `control`, `test_1` (кэшбэк 5 %), `test_2` (только push), основная метрика — оборот
за 4 недели, MDE 200 ₽, поправка Холма, CUPED по августу.

Продакт: «Ну что, раскатываем?»

А ещё «Мобильный оператор» включил тот же кэшбэк своим абонентам сам — без нас и без контроля.
Его итоги тоже лежат в базе, и продакт хочет знать, что они говорят.

У нас одна пара и один отчёт для продакта.

| Метка | Что значит |
|---|---|
| **понимать** | спросят на защите и на проверочной |
| **уметь писать** | понадобится в ДЗ-3 и в итоговом проекте |
| **знать, что существует** | достаточно помнить, что такое бывает, и загуглить |

> **Окружение** — то же, что на семинаре 7: из корня `prime-monitor`
> `uv run --project pilot jupyter lab`.

---

## 1. Данные и первая проверка

| Таблица | Одна строка — это |
|---|---|
| `pilot_candidates` | кандидат и его история до пилота |
| `pilot_assignment` | запись журнала раздачи групп — **не клиент** |
| `pilot_outcomes` | итоги клиента за 31.08–27.09: оборот, покупки, отток, обращения, тариф в конце |
| `rollout_telecom` | абонент партнёра: история, включил ли опцию, итоги |

In [ ]:
import sys
from pathlib import Path

# Окружение pilot/ не устанавливает пакет prime: у него pandas 3, а HypEx 1.0.7
# работает только с pandas 2.2. Поэтому src/ подключаем к пути сами —
# ищем папку src/prime вверх от текущей.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prime").is_dir()), None)
if root is None:
    raise RuntimeError("Не нашёл src/prime. Запустите JupyterLab из корня prime-monitor:\n"
                       "uv run --project pilot jupyter lab")
sys.path.insert(0, str(root / "src"))

import hypex

if hypex.__version__ != "1.0.7":
    raise RuntimeError(f"Нужен HypEx 1.0.7, а запущен {hypex.__version__} из {sys.executable}.\n"
                       "Закройте JupyterLab и запустите из корня prime-monitor: uv run --project pilot jupyter lab")

import warnings

import numpy as np
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine

warnings.filterwarnings("ignore")          # HypEx шумит предупреждениями pandas
pd.set_option("display.width", 160)
engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def F(obj):
    """Результаты HypEx — объекты Dataset. Для pandas-операций берём .data."""
    return obj.data if hasattr(obj, "data") else obj


print("HypEx", hypex.__version__, "· pandas", pd.__version__, "· подключились как:",
      q("select current_user as login").iloc[0, 0])

In [ ]:
# Таблицы пилота лежат в схеме prime. Если сервер не отвечает — поменяйте
# False на True: те же таблицы скачаются из репозитория курса.
SERVER_IS_DOWN = False

RAW = ("https://raw.githubusercontent.com/tikhomirovd/python-for-ba-hse-2026/master/"
       "%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/%D0%BF%D0%B8%D0%BB%D0%BE%D1%82/")


def load(name: str) -> pd.DataFrame:
    """Таблица пилота целиком; client_id — строкой, как в любой выгрузке."""
    df = pd.read_csv(RAW + f"{name}.csv.gz") if SERVER_IS_DOWN else q(f"select * from prime.{name}")
    if "client_id" in df:
        df["client_id"] = df["client_id"].astype(str)
    return df

In [ ]:
cand = load("pilot_candidates")
asg = load("pilot_assignment")
out = load("pilot_outcomes")

print(len(cand), "кандидатов ·", len(asg), "записей журнала ·", len(out), "итогов")

Записей журнала на 55 больше, чем клиентов. Кто-то записан дважды — найдём кого.

In [ ]:
# Сколько разных групп у каждого клиента в журнале?
n_grp = asg.groupby("client_id")["grp"].ЗАПОЛНИТЕ
both = n_grp[n_grp > 1].index

print(len(both), "клиентов в двух группах")
asg[asg["client_id"].isin(both)].sort_values(["client_id", "assigned_at"]).head(6)

**55 клиентов после перевыкладки сервиса 2 сентября получили вторую запись — `test_2`.**
Им ушёл push, хотя группа у них была другая. Чистым сравнение по ним уже не будет.

Сбой случился по времени раздачи, а не по свойствам клиента, поэтому выбросить их можно —
и обязательно написать об этом в отчёте. Другой честный вариант — оставить в первой
назначенной группе; эффект тогда чуть разбавится.

In [ ]:
clean = asg[~asg["client_id"].isin(both)]
assert clean["client_id"].is_unique

# validate= проверяет кратность соединения: один к одному. Повтор упадёт здесь, а не в отчёте.
df = (cand.merge(clean[["client_id", "grp"]], on="client_id", validate=ЗАПОЛНИТЕ)
          .merge(out, on="client_id", validate="one_to_one"))

print(len(df), "клиентов в анализе")
df["grp"].value_counts()

### SRM: не потерялись ли клиенты неслучайно

Делили по трети — по трети и должно прийти. Критерий χ² согласия сравнивает наблюдаемые
размеры с равными долями (слайд 5 лекции).

In [ ]:
from scipy import stats

sizes = df["grp"].value_counts().sort_index()
srm = stats.ЗАПОЛНИТЕ

print(sizes.to_dict())
print(f"chi2 = {srm.statistic:.2f}, p = {srm.pvalue:.3f}")

**p = 0,35 — перекоса нет.** Если бы был, считать эффект дальше было бы нельзя: сначала искать,
где потеряли клиентов.

Посмотрим, что даёт соблазнительный фильтр «оставим в тестах только тех, кто открыл push».

In [ ]:
seen = df[(df["grp"] == "control") | (df["saw_offer"] == 1)]
sizes_seen = seen["grp"].value_counts().sort_index()

print(sizes_seen.to_dict())
print("p =", stats.chisquare(sizes_seen).pvalue)

p ≈ 10⁻²⁹⁶ — SRM кричит. Фильтр по событию после воздействия ломает сравнение групп,
и SRM — первый, кто это замечает.

**Понимать.** SRM не про статистику, а про данные: где-то потерялись клиенты, и почти никогда
не случайно. Проверка — одна строка, делается до любого эффекта.

---

## 2. Эффект и доверительный интервал

Слайд 8. Разность средних и её интервал — по дисперсиям каждой группы отдельно (Уэлч).
Квантиль — из стандартной библиотеки.

In [ ]:
from statistics import NormalDist


def diff_ci(t: pd.Series, c: pd.Series, alpha: float = 0.05) -> dict:
    """Разность средних test − control, её 95 % интервал и p-значение."""
    d = t.mean() - c.mean()
    # Стандартная ошибка разности: дисперсия каждой группы делится на её размер.
    se = (t.var() / len(t) + ЗАПОЛНИТЕ) ** 0.5
    q = NormalDist().inv_cdf(1 - alpha / 2)
    p = 2 * (1 - NormalDist().cdf(abs(d / se)))
    return {"diff": d, "lo": d - q * se, "hi": d + q * se, "p": p}


g = df.groupby("grp")
pd.DataFrame({grp: diff_ci(g.get_group(grp)["spend"], g.get_group("control")["spend"])
              for grp in ("test_1", "test_2")}).T.round({"diff": 2, "lo": 2, "hi": 2, "p": 6})

**test_1: +315 ₽, интервал 161…468. test_2: +19 ₽, интервал −127…166.**

Сверимся с HypEx. Интервала в сводке нет — только разность и p-значение.

In [ ]:
from hypex import ABTest
from hypex.dataset import Dataset, InfoRole, TargetRole, TreatmentRole

ds = Dataset(
    roles={"client_id": InfoRole(str), "grp": ЗАПОЛНИТЕ, "spend": TargetRole()},
    data=df, default_role=InfoRole(),
)
res = ABTest(additional_tests=["t-test", "u-test"]).execute(ds)
F(res.resume)

Разница совпала до копейки, p-значения t-критерия — тоже. U-критерий согласен: оборот
вырос не только в среднем, но и у большинства клиентов.

**Уметь писать.** `diff_ci` — шесть строк, которые дают то, чего нет в сводке HypEx:
интервал. В отчёт идёт интервал, а не только p.

---

## 3. Стражи

Отток и обращения — числовые, к ним подходит тот же `diff_ci`. Тариф в конце окна —
категория из трёх значений, для неё χ² (слайд 9).

In [ ]:
guards = []
for metric in ("churned", "tickets"):
    for grp in ("test_1", "test_2"):
        r = diff_ci(g.get_group(grp)[metric], g.get_group("control")[metric])
        guards.append({"metric": metric, "grp": grp, "control": g.get_group("control")[metric].mean(), **r})

pd.DataFrame(guards).round(4)

In [ ]:
# chi2 в HypEx работает только на колонке-категории. Какой тип нужен?
df["tariff_end"] = df["tariff_end"].astype(ЗАПОЛНИТЕ)

ds_tariff = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "tariff_end": TargetRole(str)},
    data=df, default_role=InfoRole(),
)
print(F(ABTest(additional_tests=["chi2-test"]).execute(ds_tariff).resume))

# Тот же критерий в scipy — по таблице частот control против test_1.
table = pd.crosstab(df["grp"], df["tariff_end"]).loc[["control", "test_1"]]
print("scipy, test_1: p =", round(stats.chi2_contingency(table).pvalue, 4))
pd.crosstab(df["grp"], df["tariff_end"], normalize="index").round(4)

**HypEx и scipy разошлись: 0,033 против 0,0011.** `chi2-test` в HypEx 1.0.7 перед расчётом
пересчитывает частоты каждой группы на долю другой — фактически вдвое уменьшает выборку, и критерий
теряет мощность. Вывод здесь тот же — различие значимо, — но на границе он может перевернуться.
Для категорий считаем по таблице частот в scipy, HypEx — для сверки.

**У test_1 сработали два стража из трёх.** Обращений на 49 % больше (0,055 → 0,082 на клиента),
на Расширенный перешли 1,48 % вместо 2,66 %. Отток не изменился. У test_2 стражи в норме.

**Понимать.** Страж — не «дополнительная информация», а часть решающего правила. Рост основной
метрики при сработавшем страже — это «дорабатываем и повторяем», а не «раскатываем».

---

## 4. Поправка на множественность

2 группы × 4 метрики = 8 сравнений, вероятность хотя бы одного ложного — 34 %. Метод поправки —
в дизайне: Холм. В HypEx 1.0.7 его передают **членом перечисления** (слайд 18).

In [ ]:
from hypex.utils.enums import ABNTestMethodsEnum

roles_m = {"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
           "churned": TargetRole(), "tickets": TargetRole(), "purchases": TargetRole()}

res_holm = ABTest(multitest_method=ABNTestMethodsEnum.holm).execute(
    Dataset(roles=dict(roles_m), data=df, default_role=InfoRole()))
res_bonf = ABTest(multitest_method=ABNTestMethodsEnum.ЗАПОЛНИТЕ).execute(
    Dataset(roles=dict(roles_m), data=df, default_role=InfoRole()))
res_str = ABTest(multitest_method="bonferroni").execute(
    Dataset(roles=dict(roles_m), data=df, default_role=InfoRole()))

mt = F(res_holm.multitest)[["field", "group", "old p-value", "new p-value", "H0 rejected"]]
mt = mt.rename(columns={"new p-value": "holm"})
mt["bonferroni"] = F(res_bonf.multitest)["new p-value"].to_numpy()
mt["строка 'bonferroni'"] = F(res_str.multitest)["new p-value"].to_numpy()
mt.round(6)

Колонка «строка 'bonferroni'» совпала с Холмом, а не с Бонферрони: строку HypEx 1.0.7
молча игнорирует. Ошибки нет — заметить можно только так, сравнив.

Здесь выводы при любой поправке одинаковы: у test_1 значимы оборот, покупки и обращения,
у test_2 — ничего. Но на граничном p разные поправки дают разные ответы, поэтому метод
выбирают до итогов.

**Знать, что существует.** `ABNTestMethodsEnum` — Бонферрони, Холм, Бенджамини—Хохберг
(`fdr_bh`) и другие из `statsmodels`.

---

## 5. CUPED руками и в HypEx

Слайд 13, код — слайд 37 в приложении:

$$Y_{\text{cuped}} = Y - \theta\,(X - \bar X),\qquad \theta = \frac{\operatorname{cov}(Y, X)}{\operatorname{var}(X)}$$

$X$ — оборот в августе, до назначения групп.

In [ ]:
y, x = df["spend"], df["spend_prev"]

theta = ЗАПОЛНИТЕ
df["spend_cuped"] = y - theta * (x - x.mean())

print(f"theta = {theta:.3f}, дисперсия ниже на {1 - df['spend_cuped'].var() / y.var():.1%}")
g = df.groupby("grp")
pd.DataFrame({grp: diff_ci(g.get_group(grp)["spend_cuped"], g.get_group("control")["spend_cuped"])
              for grp in ("test_1", "test_2")}).T.round({"diff": 2, "lo": 2, "hi": 2, "p": 6})

In [ ]:
from hypex.dataset import PreTargetRole

ds_cuped = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
           "spend_prev": PreTargetRole(parent="spend", lag=ЗАПОЛНИТЕ)},
    data=df[["client_id", "grp", "spend", "spend_prev"]], default_role=InfoRole(),
)
res_cuped = ABTest(cuped_features={"spend": "spend_prev"}).execute(ds_cuped)
print(F(res_cuped.cuped.resume))
F(res_cuped.resume)

**Совпало: 39,2 %, +240 и +35 ₽.** Интервал test_1 сузился с 307 ₽ до 241, у test_2 верхняя
граница опустилась до 149 — ниже MDE.

Оценка test_1 сдвинулась с 315 до 240: в августе эта группа случайно тратила на 114 ₽ больше
контроля, и CUPED это учёл. Истина по генератору — +197 ₽, но по одному прогону о точности
судить нельзя: CUPED делает оценку устойчивее в среднем по возможным делениям.

### Ловушка: ковариата после начала пилота

In [ ]:
# Намеренная ошибка: ковариата — покупки В СЕНТЯБРЕ, на них кэшбэк уже повлиял.
th_leak = y.cov(df["purchases"]) / df["purchases"].var()
df["spend_leak"] = y - th_leak * (df["purchases"] - df["purchases"].mean())

print(f"дисперсия ниже на {1 - df['spend_leak'].var() / y.var():.0%}")
diff_ci(df.loc[df["grp"] == "test_1", "spend_leak"], df.loc[df["grp"] == "control", "spend_leak"])

Дисперсия упала на 70 % — и вместе с ней ушли пять шестых эффекта: кэшбэк менял число покупок,
поправка вычла его. p-значение 0,38 — «эффекта нет».

**Понимать.** Ковариата CUPED измерена **строго до** назначения групп. Это единственное условие,
при котором поправка не трогает эффект, и проверить его по данным нельзя — только по смыслу.

> **Зачем это в работе.** «Возьмём в ковариату всё, что коррелирует» — частая и дорогая ошибка.
> Корреляция с метрикой у ковариаты, затронутой воздействием, всегда отличная.

---

## 6. Квазиэксперимент: партнёр включил кэшбэк сам

Слайды 23–27. Назначение не случайно: опцию активировали сами абоненты.

In [ ]:
roll = load("rollout_telecom")
m = roll.groupby("treated")[["age", "spend_prev2", "spend_prev", "spend"]].mean()
m.round(1)

In [ ]:
naive = m.loc[1, "spend"] - m.loc[0, "spend"]
prepost = m.loc[1, "spend"] - m.loc[1, "spend_prev"]

# Разность разностей: изменение у включивших минус изменение у остальных.
did = (m.loc[1, "spend"] - m.loc[1, "spend_prev"]) - (ЗАПОЛНИТЕ)

# Плацебо: та же DiD на двух периодах ДО включения. Должна быть около нуля.
placebo = (m.loc[1, "spend_prev"] - m.loc[1, "spend_prev2"]) - (m.loc[0, "spend_prev"] - m.loc[0, "spend_prev2"])

pd.Series({"наивная разность": naive, "до-после у включивших": prepost,
           "DiD": did, "DiD-плацебо, июль → август": placebo}).round(0)

**Плацебо дало +446 вместо нуля — параллельности нет.** Опцию включали те, у кого август
выдался удачным, и в сентябре они вернулись к обычному уровню. DiD принимает это возвращение
за отрицательный эффект.

### Мэтчинг

Близнецы подбираются **в обе стороны**: включившим — из не включивших (ATT), не включившим —
из включивших (ATC); вместе — ATE. Пространство подбора — признаки, известные до сентября.

In [ ]:
from hypex import Matching
from hypex.dataset import FeatureRole

feats = ["age", "tenure_days", "kyc_level", "members", "spend_prev", "sessions_prev"]
roles_r = {"client_id": InfoRole(str), "treated": TreatmentRole(), "spend": TargetRole(),
           **{f: ЗАПОЛНИТЕ for f in feats}}

res_m = Matching().execute(Dataset(roles=dict(roles_r), data=roll, default_role=InfoRole()))
F(res_m.resume)

In [ ]:
quality = F(res_m.quality_results)

# Направление 0 — близнецы не включившим, 1 — включившим. Где пары хуже?
quality["direction"] = quality["group"].str[0]
quality.pivot_table(index="feature", columns="direction",
                    values=["TTest p-value", "KSTest p-value"]).round(4)

В направлении 0 провалены возраст и прошлый оборот: 6 996 не включившим ищут близнецов среди
2 004 — пары хуже, чем в обратную сторону. Поэтому ATC оценён хуже ATT.

Один близнец даёт широкий интервал. Пять — уже, ценой чуть более далёких пар:

In [ ]:
res_m5 = Matching(n_neighbors=ЗАПОЛНИТЕ).execute(Dataset(roles=dict(roles_r), data=roll, default_role=InfoRole()))
F(res_m5.resume)

**ATT 264, ATC 197, ATE 212 ₽** — против истинных 257, 183 и 200. Наивная разность была 1 511.

Теперь уберём из признаков прошлый оборот и сессии — то, по чему абоненты и отбирались:

In [ ]:
short = {k: v for k, v in roles_r.items() if k not in ("spend_prev", "sessions_prev")}
F(Matching().execute(Dataset(roles=short, data=roll, default_role=InfoRole())).resume)

**ATE 1 385 — почти наивная разность.** Мэтчинг учитывает смещение только по тем признакам,
которые ему дали. Признак, по которому шёл отбор, не попал в пространство подбора — смещение
осталось целиком, а интервал при этом узкий и уверенный.

**Понимать.** Мэтчинг — оценщик, а не дизайн. Он учитывает **небольшие** смещения по
**наблюдаемым** признакам. Ненаблюдаемый фактор подбор не устраняет, и проверить его
отсутствие по данным нельзя.

### Тот же пилот — мэтчингом

Мэтчинг — оценщик, и к данным рандомизированного пилота он тоже применим. Оценим им наш
пилот, control против test_1, на тех же признаках.

In [ ]:
rct = df[df["grp"].isin(["control", "test_1"])].assign(t=lambda d: (d["grp"] == "test_1").astype(int))
F(Matching().execute(Dataset(
    roles={"client_id": InfoRole(str), "t": TreatmentRole(), "spend": TargetRole(),
           **{f: FeatureRole() for f in feats}},
    data=rct, default_role=InfoRole())).resume)

ATE 347, интервал 163…531 — согласен с разностью средних (315), но не точнее. При случайном
назначении группы уже сопоставимы, и подбор добавляет свой шум. В А/Б-тесте оценщик по умолчанию —
разность средних или CUPED; мэтчинг — когда случайное деление оставило заметный перекос
по важному признаку.

---

## 7. Отчёт для продакта

Слайды 30–31. Четыре абзаца своими словами, с числами из ноутбука. Дважды щёлкните по ячейке
и замените многоточия.

**Что было зафиксировано до запуска:** …

**Результаты** — эффект с интервалом, стражи: …

**Чего мы не знаем:** …

**Решение** — по каждой группе и по запуску партнёра: …

---

## 8. Компонент: `src/prime/pilot.py`

Дополните файл с семинара 7 тремя функциями: `diff_ci` из раздела 2, `cuped` — поправка
из раздела 5, оформленная функцией, — и проверка SRM. Чтобы модуль работал и в основном
окружении `prime-monitor`, где нет scipy, SRM для двух и трёх групп считаем через стандартную
библиотеку: при $k-1$ степенях свободы

* две группы: $p = \operatorname{erfc}\bigl(\sqrt{\chi^2/2}\bigr)$;
* три группы: $p = e^{-\chi^2/2}$.

```python
import math


def srm_pvalue(sizes) -> float:
    # p-значение chi2 согласия с равными долями для 2 или 3 групп
    n = sum(sizes)
    exp = n / len(sizes)
    chi2 = sum((s - exp) ** 2 / exp for s in sizes)
    if len(sizes) == 2:
        return math.erfc(math.sqrt(chi2 / 2))
    if len(sizes) == 3:
        return math.exp(-chi2 / 2)
    raise ValueError("для 4 и более групп — scipy.stats.chisquare")


def cuped(y, x):
    # метрика после поправки CUPED и theta; y и x — pandas.Series одной длины
    theta = y.cov(x) / x.var()
    return y - theta * (x - x.mean()), theta
```

Сверьте `srm_pvalue(sizes.tolist())` с `stats.chisquare` из раздела 1 — это ★1 ниже.

```
git add src/prime/pilot.py && git commit -m "интервал, SRM и CUPED с семинара 8"
```

---

## Что унести с сегодняшнего дня

1. **До эффекта — проверки:** повторы в журнале, SRM. Фильтр «среди увидевших» ломает сравнение,
   и SRM это ловит.
2. **Интервал, а не только p.** `diff_ci` — шесть строк; в сводке HypEx интервала нет.
3. **Стражи — часть решения.** Значимый рост оборота при сработавших стражах — «доработать».
4. **CUPED сузил интервал на 22 %** — как полуторная выборка. Ковариата — строго до назначения:
   иначе исчезает эффект.
5. **Незначимо — не значит «эффекта нет».** Интервал push с CUPED −78…149 ₽ исключает эффект
   от MDE и выше, меньший — нет.
6. **Без рандомизации:** DiD проверяют плацебо, мэтчинг — качеством подбора по обоим
   направлениям и составом признаков.

---

## Хвост

Дальше — работа с открытым концом: сколько успеете на паре, остальное дома. Ничего из этого
не сдаётся. Всё это было на лекции, и всё пригодится в ДЗ-3.

### Хвост 1. CUPAC: ковариата из модели

Ковариата — прогноз модели по обороту и сессиям. Модель учится на переходе июль → август
и применяется к переходу август → сентябрь — слайд 14.

In [ ]:
from hypex.dataset import FeatureRole

roles_cupac = {
    "client_id": InfoRole(str), "grp": TreatmentRole(),
    "spend": TargetRole(cofounders=["sessions"]),
    "spend_prev": PreTargetRole(parent="spend", lag=1),
    "sessions_prev": FeatureRole(parent="sessions", lag=1),
    # Второй срез истории: без него CUPAC падает с AttributeError про NoneType.
    "spend_prev2": PreTargetRole(parent="spend", lag=ЗАПОЛНИТЕ),
    "sessions_prev2": FeatureRole(parent="sessions", lag=2),
}
cols = list(roles_cupac)
res_cupac = ABTest(enable_cupac=True, cupac_models=["linear", "ridge", "lasso"],
                   multitest_method=ABNTestMethodsEnum.holm).execute(
    Dataset(roles=roles_cupac, data=df[cols], default_role=InfoRole()))
print(F(res_cupac.cupac.resume))
F(res_cupac.resume)

**48,1 % фактического снижения против 39,2 % у CUPED.** Сессии измеряют тот же уровень клиента
другим способом — модели есть что добавить к прошлому обороту.

`variance_reduction_cv` — оценка на кросс-валидации при обучении, `variance_reduction_real` —
фактическая. Если вторая заметно ниже первой, аудитория между периодами изменилась
или модель переобучилась.

### Хвост 2. Статистическая значимость и деньги

Слайд «Статистическая значимость — не практическая». Ставки комиссии и цены обращения в базе нет —
задаём **допущениями**, как на С6.

In [ ]:
COMMISSION = 0.015        # допущение: комиссия с оборота партнёров, как на С6
TICKET_COST = 150         # допущение: стоимость одного обращения, ₽
UPGRADE_VALUE = 200       # разница цен Расширенного и Базового, ₽ в месяц

c, t = df[df["grp"] == "control"], df[df["grp"] == "test_1"]
lift = diff_ci(t["spend"], c["spend"])
upgrade_gap = (t["tariff_end"] == "Расширенный").mean() - (c["tariff_end"] == "Расширенный").mean()
ticket_gap = t["tickets"].mean() - c["tickets"].mean()

money = pd.Series({
    "комиссия с прироста оборота": lift["diff"] * COMMISSION,
    "недополученные переходы": upgrade_gap * UPGRADE_VALUE,
    # Обращений стало больше — это расход. Какой знак у строки?
    "обращения": ЗАПОЛНИТЕ,
})
money["итого на клиента за 4 недели"] = money.sum()
money.round(2)

**Оборот +9,7 % с p = 0,00006 — а в деньгах минус 1,7 ₽ на клиента.** Значимость говорит,
что эффект не ноль. Окупается ли он, считают отдельно, и вывод держится на допущениях:
их пишут в отчёт рядом с выводом.

### Хвост 3. Эффект по сегментам — гипотеза, а не вывод

Продакт спросит: «А у кого кэшбэк сработал сильнее?» Посмотрим — и честно.

In [ ]:
# Эффект test_1 с CUPED отдельно по числу участников подписки (members: 0, 1, 2)
# — diff_ci по spend_cuped в каждом сегменте. Добавьте поправку Холма на три сравнения
# (statsmodels.stats.multitest.multipletests). Что из этого можно сказать продакту?
by_members = None   # ← ваш расчёт

by_members

**Что сказать продакту**

Две-три фразы своими словами: какой сегмент выглядит сильнее, значимо ли различие **между**
сегментами (подсказка: интервалы пересекаются?) и что из этого следует для следующего пилота.

…

---

## Задачи со звёздочкой

Шесть задач на то, что было на лекции. **Не сдаются и не оцениваются.** Считаются на общих
таблицах `df` и `roll` из разделов 1–6 — если перезапускали ядро, прогоните их.

In [ ]:
# Это просто запустите
ANSWERS = {
    "★1": "a976726b1d",
    "★2": "f766456ab1",
    "★3": "148422b105",
    "★4": "4e07408562",
    "★5": "342a441fd8",
    "★6": "eb8ae3c398"
}

In [ ]:
# Это тоже
import hashlib

import numpy as np
import pandas as pd


def _norm(x):
    if isinstance(x, bool):
        return repr(x)
    if isinstance(x, float):
        return f"{x:.2f}"
    if isinstance(x, int):
        return str(x)
    if isinstance(x, dict):
        return "{" + ",".join(f"{_norm(k)}:{_norm(v)}" for k, v in sorted(x.items(), key=lambda kv: _norm(kv[0]))) + "}"
    if isinstance(x, (list, tuple)):
        return "[" + ",".join(_norm(v) for v in x) + "]"
    if isinstance(x, np.integer):
        return str(int(x))
    if isinstance(x, np.floating):
        return f"{float(x):.2f}"
    if isinstance(x, np.bool_):
        return repr(bool(x))
    return repr(x)


def check(task: str, got) -> None:
    """Сверить ответ с эталоном. Эталоны лежат хешами — подсмотреть нельзя."""
    if got is None:
        print(f"{task}: не решено")
        return
    digest = hashlib.sha256(_norm(got).encode()).hexdigest()[:10]
    if not ANSWERS:
        print(f"HASH:{task}={digest}")
    elif digest == ANSWERS.get(task):
        print(f"{task}: верно")
    else:
        print(f"{task}: не сходится, получилось {got!r}")


print("проверка готова")

### ★ 1. SRM без scipy

Напишите `srm_pvalue` из раздела 8 и посчитайте p для размеров групп после чистки.
Должно совпасть со `stats.chisquare`.

**Ответ:** доля с двумя знаками — `round(x, 2)`.

In [ ]:
answer = None   # ← ваш расчёт

check("★1", answer)

### ★ 2. Что было бы без чистки журнала

Соедините журнал с итогами **как есть**, без удаления повторов. Сколько строк получится в группе
`test_2`? Именно столько клиентов посчиталось бы в ней — с 55 чужими.

**Ответ:** целое число.

In [ ]:
answer = None   # ← ваш расчёт

check("★2", answer)

### ★ 3. Во сколько раз CUPED сузил интервал — и какой выборке это равно

Отношение ширины 95 % интервала test_1 без поправки к ширине с CUPED. Возведите его в квадрат:
во столько раз пришлось бы увеличить выборку, чтобы так же сузить интервал без CUPED.

**Ответ:** список из двух чисел с двумя знаками — `[round(ratio, 2), round(ratio ** 2, 2)]`.

In [ ]:
answer = None   # ← ваш расчёт

check("★3", answer)

### ★ 4. Сколько гипотез отвергнуто

Сравнения из раздела 4 — 2 группы × 4 метрики. Сколько из восьми гипотез отвергнуто после поправки
Бонферрони? Возьмите `res_bonf` — тот, что с перечислением.

**Ответ:** целое число.

In [ ]:
answer = None   # ← ваш расчёт

check("★4", answer)

### ★ 5. Где кончается подглядывание

Слайд 19. Если смотреть на p-значение $k$ раз и останавливаться на первом $p < 0{,}05$, доля ложных
эффектов растёт. Самая грубая верхняя оценка — как для $k$ независимых сравнений: $1-0{,}95^k$.
Посчитайте её для 28 ежедневных взглядов.

Симуляция на лекции дала 29 %. Почему грубая оценка настолько больше?

**Ответ:** доля с двумя знаками — `round(x, 2)`.

In [ ]:
answer = None   # ← ваш расчёт

check("★5", answer)

### ★ 6. Плацебо на сентябре у не включивших

DiD-плацебо проверяет параллельность на прошлом. Посчитайте другое плацебо: у **не включивших**
сравните изменение август → сентябрь с изменением июль → август. Это сезон без всякого кэшбэка.

**Ответ:** список из двух чисел, рубли с двумя знаками — изменение июль → август
и август → сентябрь у не включивших.

In [ ]:
answer = None   # ← ваш расчёт

check("★6", answer)